# GA1-220501094-01-AA1-EV09 – Proyecto final: informe analítico
## Zona Fit Evolution: análisis de deserción de miembros

**Aprendiz:** Juan Diego Mejía Llano
**Programa:** Análisis y Desarrollo de Software (ADSO) – SENA Regional Antioquia, ficha 3223873
**Fecha:** noviembre de 2026

Este notebook hace el análisis descriptivo y visual de las tablas preparadas en la EV07 para responder la pregunta analítica del proyecto. Todas las cifras del informe [`reports/analytical_report.md`](../reports/analytical_report.md) salen de las salidas de este notebook. El uso de inteligencia artificial está documentado en [`prompts.md`](../prompts.md).

**Contenido**

1. Planteamiento: pregunta analítica y subpreguntas (Momento 2)
2. Diccionario de datos (Momento 2)
3. Carga y validación de los datos (Momento 3)
4. Análisis univariado (Momento 4)
5. Análisis bivariado y por segmentos (Momento 5)
6. Hallazgos, respuesta a la pregunta y recomendaciones (Momento 6)

# 1. Planteamiento (Momento 2)

**Problema:** Zona Fit Evolution necesita saber qué hace que un miembro deje de renovar su membresía, para actuar a tiempo con los miembros en riesgo.

**Pregunta analítica:** ¿Qué factores (tipo de plan, frecuencia de asistencia, horario de entrenamiento y antigüedad) influyen en que un miembro de Zona Fit Evolution cancele o no renueve su membresía durante sus primeros seis meses?

**Tipo de análisis:** descriptivo, con un componente diagnóstico (asociación entre factores y deserción).

**Cómo se mide la deserción**

- **Por membresía (`no_renovo`):** una membresía no se renueva si fue cancelada o si pasaron más de 30 días desde su fin sin una nueva compra. Solo se usan las membresías `observable == True`, cuyo resultado ya se conoce al corte (15/10/2026).
- **Por miembro (`desertor_6m`):** el miembro deja de renovar dentro de sus primeros 180 días. Solo se usan los miembros `evaluable == True`, cuya fecha de alta tiene al menos 180 + 30 días al corte.

**Subpreguntas**

| # | Subpregunta | Datos | Operación | Gráfica |
|---|---|---|---|---|
| 1 | ¿Cómo varía la deserción según el plan? | `nombre_plan`, `tipo_plan`, `no_renovo` · `tipo_plan_inicial`, `desertor_6m` | `groupby` + `agg` | Barras |
| 2 | ¿Qué relación hay entre la frecuencia semanal de asistencia y la deserción? | `visitas_semana`, `frecuencia`, `no_renovo` · `frecuencia_inicial`, `desertor_6m` | `groupby`, boxplot | Barras ordenadas y boxplot |
| 3 | ¿La franja horaria se asocia con la deserción? | `franja_principal`, `no_renovo` · `franja_inicial`, `desertor_6m` | `crosstab` normalizado | Barras agrupadas |
| 4 | ¿Cómo cambia el riesgo según la antigüedad y cuántos miembros desertan en sus primeros 6 meses? | `tramo_antiguedad`, `antiguedad_dias`, `no_renovo` · `desertor_6m` | `groupby`, correlación | Barras y dispersión |
| 5 | ¿Cómo evoluciona la tasa de no renovación según el mes de inicio de la membresía? | `fecha_inicio`, `no_renovo` (solo observables) | `groupby` por mes | Líneas |

Las subpreguntas se validaron contra las columnas reales (apartado 1 de la bitácora): `tipo_plan` solo tiene dos valores, así que el detalle se hace con `nombre_plan`; las membresías sin visitas no tienen franja y se excluyen en la subpregunta 3; y los últimos meses de la serie tienen pocas membresías observables (censura al corte).

# 2. Diccionario de datos (Momento 2)

Tablas generadas en la EV07 (`data/processed/`). Los tipos estadísticos siguen la clasificación de la EV08.

### `zonafit_memberships_analytic` (una fila por membresía)

| variable | descripción | tipo en Pandas | tipo estadístico | unidad |
|---|---|---|---|---|
| id_membresia | Identificador de la membresía | int16 | Cualitativa nominal | — |
| id_miembro | Identificador del miembro que compró la membresía | int16 | Cualitativa nominal | — |
| id_plan | Identificador del plan comprado | int8 | Cualitativa nominal | — |
| fecha_inicio | Primer día de vigencia de la membresía | datetime64[us] | Cuantitativa continua | fecha |
| fecha_fin | Último día de vigencia de la membresía | datetime64[us] | Cuantitativa continua | fecha |
| precio_pagado | Valor cobrado por la membresía (puede tener descuento) | int64 | Cuantitativa continua | COP |
| estado | Estado al corte: Activa, Vencida o Cancelada | category | Cualitativa nominal | — |
| siguiente_inicio | Inicio de la siguiente membresía del mismo miembro (vacía si no hay) | datetime64[us] | Cuantitativa continua | fecha |
| dias_hasta_siguiente | Días entre el fin de esta membresía y el inicio de la siguiente | float64 | Cuantitativa discreta | días |
| renovada | True si compró otra membresía en los 30 días siguientes al fin y no la canceló | bool | Cualitativa nominal | — |
| observable | True si ya se conoce el resultado de renovación al corte | bool | Cualitativa nominal | — |
| no_renovo | True si la membresía es observable y no se renovó | bool | Cualitativa nominal | — |
| numero_membresia | Orden de la membresía en el historial del miembro (1 = primera) | int64 | Cuantitativa discreta | membresías |
| antiguedad_dias | Días desde la primera membresía del miembro hasta el inicio de esta | int64 | Cuantitativa discreta | días |
| nombre_plan | Nombre comercial del plan | str | Cualitativa nominal | — |
| tipo_plan | Ilimitado o Tiquetera | category | Cualitativa nominal | — |
| duracion_dias | Días de vigencia del plan | int16 | Cuantitativa discreta | días |
| precio | Precio de lista del plan | int32 | Cuantitativa continua | COP |
| genero | Género del miembro (M, F o Sin dato) | category | Cualitativa nominal | — |
| fecha_nacimiento | Fecha de nacimiento del miembro | datetime64[us] | Cuantitativa continua | fecha |
| comuna | Comuna de Medellín donde vive el miembro (o Sin dato) | category | Cualitativa nominal | — |
| canal_captacion | Instagram, Referido, Walk-in, Volante o Convenio empresarial | category | Cualitativa nominal | — |
| fecha_registro | Fecha de inscripción del miembro | datetime64[us] | Cuantitativa continua | fecha |
| pagado_aprobado | Suma de pagos aprobados de la membresía | int64 | Cuantitativa continua | COP |
| pagado_pendiente | Suma de pagos pendientes de la membresía | int64 | Cuantitativa continua | COP |
| pagado_rechazado | Suma de pagos rechazados de la membresía | int64 | Cuantitativa continua | COP |
| numero_pagos | Cantidad de pagos registrados para la membresía | int64 | Cuantitativa discreta | pagos |
| visitas_mañana | Ingresos válidos entre las 5:00 y las 11:59 | int16 | Cuantitativa discreta | visitas |
| visitas_tarde | Ingresos válidos entre las 12:00 y las 17:59 | int16 | Cuantitativa discreta | visitas |
| visitas_noche | Ingresos válidos entre las 18:00 y las 21:59 | int16 | Cuantitativa discreta | visitas |
| visitas | Total de ingresos válidos durante la membresía | int16 | Cuantitativa discreta | visitas |
| primera_visita | Fecha y hora del primer ingreso de la membresía | datetime64[us] | Cuantitativa continua | fecha y hora |
| ultima_visita | Fecha y hora del último ingreso de la membresía | datetime64[us] | Cuantitativa continua | fecha y hora |
| visitas_semana | Promedio de visitas por semana durante la vigencia (hasta el corte) | float64 | Cuantitativa continua | visitas/semana |
| frecuencia | Rango de visitas semanales (hasta 1, más de 1 a 2, más de 2 a 3, más de 3) | category | Cualitativa ordinal | — |
| franja_principal | Franja con más ingresos: mañana, tarde o noche (vacía si no hubo visitas) | str | Cualitativa ordinal | — |
| tramo_antiguedad | 0 a 6 meses, 6 a 12 meses, más de 12 meses | category | Cualitativa ordinal | — |
| edad_al_inicio | Edad en años cumplidos al iniciar la membresía | Int16 | Cuantitativa discreta | años |
| pago_completo | True si los pagos aprobados cubren el precio pagado | bool | Cualitativa nominal | — |

### `zonafit_members_analytic` (una fila por miembro)

| variable | descripción | tipo en Pandas | tipo estadístico | unidad |
|---|---|---|---|---|
| id_miembro | Identificador del miembro | int16 | Cualitativa nominal | — |
| documento_hash | Documento de identidad anonimizado | str | Cualitativa nominal | — |
| genero | Género registrado (M, F o Sin dato) | category | Cualitativa nominal | — |
| fecha_nacimiento | Fecha de nacimiento | datetime64[us] | Cuantitativa continua | fecha |
| comuna | Comuna de Medellín donde vive (o Sin dato) | category | Cualitativa nominal | — |
| canal_captacion | Canal por el que llegó al gimnasio | category | Cualitativa nominal | — |
| fecha_registro | Fecha de inscripción | datetime64[us] | Cuantitativa continua | fecha |
| membresias | Número de membresías compradas | int64 | Cuantitativa discreta | membresías |
| ultima_fecha_fin | Fecha de fin de su membresía más reciente | datetime64[us] | Cuantitativa continua | fecha |
| membresias_no_renovadas | Número de membresías que no renovó | int64 | Cuantitativa discreta | membresías |
| visitas_totales | Total de ingresos válidos en todo su historial | int16 | Cuantitativa discreta | visitas |
| total_pagado | Suma de lo pagado en todas sus membresías | int64 | Cuantitativa continua | COP |
| plan_inicial | Nombre del plan de su primera membresía | str | Cualitativa nominal | — |
| tipo_plan_inicial | Tipo del plan de su primera membresía | category | Cualitativa nominal | — |
| visitas_semana_inicial | Visitas promedio por semana en su primera membresía | float64 | Cuantitativa continua | visitas/semana |
| franja_inicial | Franja principal de su primera membresía | str | Cualitativa ordinal | — |
| fecha_alta | Fecha de inicio de su primera membresía | datetime64[us] | Cuantitativa continua | fecha |
| primera_no_renovacion | Fecha de su primera membresía no renovada (vacía si siempre renovó) | datetime64[us] | Cuantitativa continua | fecha |
| evaluable | True si ya pasaron 180 + 30 días desde el alta (se puede evaluar la deserción temprana) | bool | Cualitativa nominal | — |
| desertor_6m | True si dejó de renovar dentro de los primeros 180 días desde el alta | bool | Cualitativa nominal | — |
| frecuencia_inicial | Rango de visitas semanales en su primera membresía | category | Cualitativa ordinal | — |
| edad_al_alta | Edad en años cumplidos en la fecha de alta | Int16 | Cuantitativa discreta | años |
| activo_al_corte | True si su membresía más reciente sigue vigente al 15/10/2026 | bool | Cualitativa nominal | — |

**Clasificaciones discutibles**

- **Fechas:** el tiempo es continuo; en el análisis se agrupan por mes para la serie temporal.
- **Montos en COP:** son enteros porque el peso no usa centavos, pero el dinero se mide, no se cuenta: se tratan como continuos.
- **Edad y columnas en días:** se registran en unidades enteras (años cumplidos, días), por eso son discretas.
- **Franja:** mañana, tarde y noche siguen el orden del día; por eso es ordinal.
- **Booleanos e identificadores:** son nominales; un identificador numérico no admite operaciones aritméticas.

# 3. Carga y validación de los datos (Momento 3)

Se importan las librerías del análisis. Matplotlib se usa para las gráficas de los momentos 4 y 5.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

print(f"pandas {pd.__version__}")

pandas 3.0.5


Se cargan las dos tablas con rutas relativas al notebook y se valida que las dimensiones y los tipos coincidan con el diccionario de datos.

In [2]:
# Rutas relativas desde la carpeta notebooks/
RUTA_MEMBRESIAS = "../data/processed/zonafit_memberships_analytic.parquet"
RUTA_MIEMBROS = "../data/processed/zonafit_members_analytic.parquet"

# 1. Cargar las dos tablas analíticas (Parquet conserva los tipos de dato)
membresias = pd.read_parquet(RUTA_MEMBRESIAS)
miembros = pd.read_parquet(RUTA_MIEMBROS)

# 2. Validar dimensiones esperadas
assert membresias.shape == (3425, 39), f"membresias tiene {membresias.shape}, se esperaba (3425, 39)"
assert miembros.shape == (900, 23), f"miembros tiene {miembros.shape}, se esperaba (900, 23)"

# 3. Tipos esperados según el diccionario de datos
tipos_membresias = {
    "id_membresia": "int16", "id_miembro": "int16", "id_plan": "int8",
    "fecha_inicio": "datetime64[us]", "fecha_fin": "datetime64[us]", "precio_pagado": "int64",
    "estado": "category", "siguiente_inicio": "datetime64[us]", "dias_hasta_siguiente": "float64",
    "renovada": "bool", "observable": "bool", "no_renovo": "bool", "numero_membresia": "int64",
    "antiguedad_dias": "int64", "nombre_plan": "str", "tipo_plan": "category", "duracion_dias": "int16",
    "precio": "int32", "genero": "category", "fecha_nacimiento": "datetime64[us]", "comuna": "category",
    "canal_captacion": "category", "fecha_registro": "datetime64[us]", "pagado_aprobado": "int64",
    "pagado_pendiente": "int64", "pagado_rechazado": "int64", "numero_pagos": "int64",
    "visitas_mañana": "int16", "visitas_tarde": "int16", "visitas_noche": "int16", "visitas": "int16",
    "primera_visita": "datetime64[us]", "ultima_visita": "datetime64[us]", "visitas_semana": "float64",
    "frecuencia": "category", "franja_principal": "str", "tramo_antiguedad": "category",
    "edad_al_inicio": "Int16", "pago_completo": "bool",
}
tipos_miembros = {
    "id_miembro": "int16", "documento_hash": "str", "genero": "category",
    "fecha_nacimiento": "datetime64[us]", "comuna": "category", "canal_captacion": "category",
    "fecha_registro": "datetime64[us]", "membresias": "int64", "ultima_fecha_fin": "datetime64[us]",
    "membresias_no_renovadas": "int64", "visitas_totales": "int16", "total_pagado": "int64",
    "plan_inicial": "str", "tipo_plan_inicial": "category", "visitas_semana_inicial": "float64",
    "franja_inicial": "str", "fecha_alta": "datetime64[us]", "primera_no_renovacion": "datetime64[us]",
    "evaluable": "bool", "desertor_6m": "bool", "frecuencia_inicial": "category",
    "edad_al_alta": "Int16", "activo_al_corte": "bool",
}

# Comparar tipo esperado vs. tipo real (sin bucles: se alinean dos Series por nombre de columna)
def comparar_tipos(df, esperados):
    comparacion = pd.DataFrame({
        "esperado": pd.Series(esperados),
        "real": df.dtypes.astype(str),
    })
    return comparacion[comparacion["esperado"] != comparacion["real"]]

diferencias_membresias = comparar_tipos(membresias, tipos_membresias)
diferencias_miembros = comparar_tipos(miembros, tipos_miembros)

print("Columnas de membresias que no coinciden con el diccionario:")
print(diferencias_membresias if not diferencias_membresias.empty else "Ninguna")
print("\nColumnas de miembros que no coinciden con el diccionario:")
print(diferencias_miembros if not diferencias_miembros.empty else "Ninguna")

assert diferencias_membresias.empty, "Los tipos de membresias no coinciden con el diccionario"
assert diferencias_miembros.empty, "Los tipos de miembros no coinciden con el diccionario"

# 4. Resumen de cada tabla
print("\n--- membresias ---")
membresias.info()
print("\n--- miembros ---")
miembros.info()


Columnas de membresias que no coinciden con el diccionario:
Ninguna

Columnas de miembros que no coinciden con el diccionario:
Ninguna

--- membresias ---
<class 'pandas.DataFrame'>
RangeIndex: 3425 entries, 0 to 3424
Data columns (total 39 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   id_membresia          3425 non-null   int16         
 1   id_miembro            3425 non-null   int16         
 2   id_plan               3425 non-null   int8          
 3   fecha_inicio          3425 non-null   datetime64[us]
 4   fecha_fin             3425 non-null   datetime64[us]
 5   precio_pagado         3425 non-null   int64         
 6   estado                3425 non-null   category      
 7   siguiente_inicio      2525 non-null   datetime64[us]
 8   dias_hasta_siguiente  2525 non-null   float64       
 9   renovada              3425 non-null   bool          
 10  observable            3425 non-null   bool      

**Interpretación:** las dos tablas cargan con las dimensiones esperadas (3.425 membresías y 900 miembros) y sus 62 columnas tienen exactamente los tipos del diccionario. Gracias al formato Parquet, las fechas llegan como `datetime64`, las categorías como `category` y la edad como entero con nulos (`Int16`), sin conversiones adicionales.

Los faltantes que muestra `info()` son esperados: `siguiente_inicio` y `dias_hasta_siguiente` están vacíos cuando no hubo una membresía posterior, `primera_visita`, `ultima_visita` y `franja_principal` cuando no hubo visitas, la edad cuando falta la fecha de nacimiento y `primera_no_renovacion` cuando el miembro siempre renovó.

## Verificación de calidad

Antes de analizar se comprueba que los problemas del diagnóstico (EV06) quedaron resueltos en la EV07 y que se cumplen las reglas del análisis.

In [3]:
FECHA_CORTE = pd.Timestamp("2026-10-15")

# 1. Estado: ninguna membresía puede seguir "Activa" si su fecha de fin ya pasó
activas_vencidas = membresias[(membresias["estado"] == "Activa") & (membresias["fecha_fin"] < FECHA_CORTE)]
assert activas_vencidas.empty, f"Hay {len(activas_vencidas)} membresías Activa ya vencidas"

# 2. Duplicados: identificadores únicos y ninguna persona inscrita dos veces
assert membresias["id_membresia"].is_unique, "Hay membresías duplicadas"
assert miembros["id_miembro"].is_unique, "Hay miembros duplicados (id_miembro)"
assert miembros["documento_hash"].is_unique, "Hay personas inscritas dos veces (documento_hash)"
# Pagos duplicados: si se repitiera un pago, lo aprobado superaría el valor cobrado
assert (membresias["pagado_aprobado"] <= membresias["precio_pagado"]).all(), "Pagos aprobados mayores al precio: posibles pagos duplicados"

# 3. Tipos de dato: fechas como datetime y montos como números
fechas_m = membresias[["fecha_inicio", "fecha_fin", "fecha_registro", "fecha_nacimiento"]]
fechas_s = miembros[["fecha_registro", "fecha_nacimiento", "fecha_alta", "ultima_fecha_fin"]]
assert fechas_m.dtypes.map(pd.api.types.is_datetime64_any_dtype).all(), "Hay fechas como texto en membresías"
assert fechas_s.dtypes.map(pd.api.types.is_datetime64_any_dtype).all(), "Hay fechas como texto en miembros"
montos = membresias[["precio_pagado", "precio", "pagado_aprobado", "pagado_pendiente", "pagado_rechazado"]]
assert montos.dtypes.map(pd.api.types.is_numeric_dtype).all(), "Hay montos como texto en membresías"
assert pd.api.types.is_numeric_dtype(miembros["total_pagado"]), "total_pagado no es numérico"

# 4. Categorías y faltantes de genero y comuna
assert set(miembros["genero"].unique()) <= {"M", "F", "Sin dato"}, "genero tiene categorías sin unificar"
assert set(membresias["genero"].unique()) <= {"M", "F", "Sin dato"}, "genero (membresías) tiene categorías sin unificar"
assert miembros[["genero", "comuna"]].notna().all().all(), "Hay faltantes sin tratar en genero o comuna (miembros)"
assert membresias[["genero", "comuna"]].notna().all().all(), "Hay faltantes sin tratar en genero o comuna (membresías)"

# 5. Reglas de negocio del análisis
assert not (membresias["no_renovo"] & ~membresias["observable"]).any(), "no_renovo es True en membresías no observables"
no_evaluables = miembros[~miembros["evaluable"]]
assert (no_evaluables["fecha_alta"] > FECHA_CORTE - pd.Timedelta(days=210)).all(), "Hay miembros no evaluables con más de 210 días desde el alta"
assert (membresias["visitas_semana"] >= 0).all(), "visitas_semana tiene valores negativos"
suma_franjas = membresias[["visitas_mañana", "visitas_tarde", "visitas_noche"]].sum(axis=1)
assert (suma_franjas == membresias["visitas"]).all(), "visitas no coincide con la suma de las tres franjas"

print("Calidad verificada: todas las reglas se cumplen.")
print(f"Miembros evaluables a 6 meses: {miembros['evaluable'].sum()}")


Calidad verificada: todas las reglas se cumplen.
Miembros evaluables a 6 meses: 671


**Interpretación:** los problemas del diagnóstico ya no aparecen. No hay membresías `Activa` vencidas, ni duplicados, ni fechas o montos como texto, y `genero` y `comuna` no tienen faltantes sin tratar. Se cumplen además las reglas del análisis.

De los 900 miembros, **671 son evaluables** a seis meses. La regla de `evaluable` es necesaria: los miembros que se inscribieron hace poco y ya desertaron también aparecen con `desertor_6m = True`, así que incluirlos inflaría la tasa de deserción temprana.

# 4. Análisis univariado (Momento 4)

*Se desarrolla en el siguiente avance.*

# 5. Análisis bivariado y por segmentos (Momento 5)

*Se desarrolla en un avance posterior.*

# 6. Hallazgos, respuesta a la pregunta y recomendaciones (Momento 6)

*Se desarrolla en un avance posterior.*